# Use Model Context Protocol (MCP) as tools with Strands Agents

## Overview
The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) is an open protocol that standardizes how applications provide context to Large Language Models (LLMs). Strands Agents SDK integrates with MCP to extend agent capabilities through external tools and services.

MCP enables communication between agents and MCP servers that provide additional tools. The Strands Agents SDK includes built-in support for connecting to MCP servers and using their tools.

In this example we will show you how to use MCP tools on your Strands Agent. We will use the [AWS Documentation MCP server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server) which provides tools to access AWS documentation, search for content, and get recommendations. This MCP server has 3 main features:

- **Read Documentation**: Fetch and convert AWS documentation pages to markdown format
- **Search Documentation**: Search AWS documentation using the official search API
- **Recommendations**: Get content recommendations for AWS documentation pages



## Tutorial Details

| Information      | Details                                         |
|:-----------------|:------------------------------------------------|
| Agent structure  | Single agent                                    |
| Model provider   | Amazon Bedrock                                  |
| Model            | Anthropic Claude Sonnet 4.5                     |
| Tools            | MCP tools                                       |
| Strands features | MCPClient, stdio and Streamable HTTP transports |

## Architecture

<div style="text-align:center">
    <img src="images/architecture.png" width="65%" />
</div>

## What you'll learn
* Connect an agent to an MCP server over stdio
* Connect over Streamable HTTP
* Use tools from multiple MCP servers with one agent
* Invoke MCP tools directly

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock

Let's now install the requirement packages for our agent

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
import threading
import time
from datetime import timedelta

from mcp import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamablehttp_client
from mcp.server import FastMCP
from strands import Agent
from strands.tools.mcp import MCPClient

### Connect to MCP server using stdio transport

[Transports](https://modelcontextprotocol.io/specification/2025-03-26/basic/transports) in MCP provide the foundations for communication between clients and servers. It handles the underlying mechanics of how messages are sent and received. At the moment there are three standards transport implementations built-in in MCP:

- **Standard Input/Output (stdio)**: enables communication through standard input and output streams. It is particularly useful for local integrations and command-line tools
- **Streamable HTTP**: this replaces the HTTP+SSE transport from previous protocol version. In the Streamable HTTP transport, the server operates as an independent process that can handle multiple client connections. This transport uses HTTP POST and GET requests. Server can optionally make use of Server-Sent Events (SSE) to stream multiple server messages. This permits basic MCP servers, as well as more feature-rich servers supporting streaming and server-to-client notifications and requests.
- **SSE**: legacy transport for HTTP-based MCP servers that use Server-Sent Events transport  

Overall, you should use stdio for building command-line tools, implementing local integrations and working with shell scripts. You should use Streamable HTTP transports when you need a flexible and efficient way for AI agents to communicate with tools and services, especially when dealing with stateless communication or when minimizing resource usage is crucial.

You can also use **custom transports** implementation for your specific needs. 


Let's now connect to the MCP server using stdio transport. First of all, we will use the class `MCPClient` to connect to the [AWS Documentation MCP Server](https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server). This server provides tools to access AWS documentation, search for content, and get recommendations.

In [ ]:
# Connect to an MCP server using stdio transport
stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `stdio_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [ ]:
# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent("What is Amazon Bedrock pricing model. Be concise.")

### Connect to MCP server using Streamable HTTP

Let's now connect to the MCP server using Streamable HTTP transport. First let's start a simple MCP server using Streamable HTTP transport. 

For this example we will create our own MCP server. The architecture will look as following

<div style="text-align:center">
    <img src="images/architecture_2.png" width="65%" />
</div>

In [ ]:
# Create an MCP server
mcp = FastMCP("Calculator Server")

# Define a tool


@mcp.tool(description="Calculator tool which performs calculations")
def calculator(x: int, y: int) -> int:
    return x + y


@mcp.tool(description="This is a long running tool")
def long_running_tool(name: str) -> str:
    time.sleep(25)
    return f"Hello {name}"


def main():
    mcp.run(transport="streamable-http", mount_path="mcp")

Let's now start a thread with the `streamable-http` server

In [ ]:
thread = threading.Thread(target=main)
thread.start()

#### Integrating Streamable HTTP client with Agent

Now let's use `streamablehttp_client` integrate this server with a simple agent. 

In [ ]:
def create_streamable_http_transport():
    return streamablehttp_client("http://localhost:8000/mcp")


streamable_http_mcp_client = MCPClient(create_streamable_http_transport)

#### Setup agent configuration and invoke it

Next we will set our agent configuration using the tools from the `streamable_http_mcp_client` object we just created. To do so, we need to list the tools available in the MCP server. We can use the `list_tools_sync` method for it. 

After that, we will ask a question to our agent.

In [ ]:
with streamable_http_mcp_client:
    tools = streamable_http_mcp_client.list_tools_sync()

    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = str(agent("What is 2 + 2?"))

### Direct Tool Invocation

While tools are typically invoked by the agent based on user requests, you can also call MCP tools directly. This can be useful for workflow scenarios where you orchestrate multiple tools together.

In [ ]:
query = {"x": 10, "y": 20}

with streamable_http_mcp_client:
    # direct tool invocation
    result = streamable_http_mcp_client.call_tool_sync(
        tool_use_id="tool-123", name="calculator", arguments=query
    )

    # Process the result
    print(f"Calculation result: {result['content'][0]['text']}")

You can optionally also provide `read_timeout_seconds` while calling an MCP server tool to avoid it running for too long

In [ ]:
with streamable_http_mcp_client:
    try:
        result = streamable_http_mcp_client.call_tool_sync(
            tool_use_id="tool-123",
            name="long_running_tool",
            arguments={"name": "Amazon"},
            read_timeout_seconds=timedelta(seconds=30),
        )

        if result["status"] == "error":
            print(f"Tool execution failed: {result['content'][0]['text']}")
        else:
            print(f"Tool execution succeeded: {result['content'][0]['text']}")
    except Exception as e:
        print(f"Tool call timed out or failed: {str(e)}")

### Interacting with multiple MCP servers

With Strands Agents you can also interact with multiple MCP servers using the same agent. Tool execution strategy is configured with the `tool_executor` parameter: agents use `ConcurrentToolExecutor` by default, which runs the model's tool calls in parallel, and `SequentialToolExecutor` runs them one at a time. Let's create a new agent that uses the tools from both servers:

<div style="text-align:center">
    <img src="images/architecture_3.png" width="85%" />
</div>

In this agent, we will again use the AWS Documentation MCP server and we will also use the [AWS IaC MCP Server](https://awslabs.github.io/mcp/servers/aws-iac-mcp-server) which helps with AWS Cloud Development Kit (CDK) best practices, CloudFormation validation, infrastructure as code patterns and security compliance.

First let's connect to the two MCP servers using the stdio transport

In [ ]:
# Connect to an MCP server using stdio transport
aws_docs_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command="uvx", args=["awslabs.aws-documentation-mcp-server@latest"]
        )
    )
)

# Connect to the AWS IaC MCP server using stdio transport
iac_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command="uvx", args=["awslabs.aws-iac-mcp-server@latest"])
    )
)

#### Create Agent with MCP servers

Next we will create the agent with the tools from both MCP servers

In [ ]:
# Create an agent with MCP tools
with aws_docs_mcp_client, iac_mcp_client:
    # Get the tools from the MCP server
    tools = aws_docs_mcp_client.list_tools_sync() + iac_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        tools=tools)

    response = agent(
        "What is Amazon Bedrock pricing model. Be concise. Also what are the best practices related to CDK?"
    )

### MCP is more than tools

This notebook connected to MCP servers to use their **tools**, but tools are only one of the three capabilities an MCP server can expose. Servers can also offer **prompts**, which are reusable, parameterized message templates, and **resources**, which are read-only data addressed by a URI, such as a file or a configuration value. The same `MCPClient` can discover and read them through `list_prompts_sync` and `get_prompt_sync` for prompts, and `list_resources_sync` and `read_resource_sync` for resources. To learn when to reach for each, see the MCP documentation on [prompts](https://modelcontextprotocol.io/docs/concepts/prompts) and [resources](https://modelcontextprotocol.io/docs/concepts/resources). The Strands [`MCPClient` API reference](https://strandsagents.com/docs/api/python/strands.tools.mcp.mcp_client/) documents each of these methods.

## Summary

In this notebook you learned how to connect with MCP servers using Strands Agents and two MCP transport protocols: stdio and Streamable HTTP. You also learned how to connect multiple MCP servers to the same agent. Next, let's see how to use different models with your agent.

---

# Exercise — Put your own tool behind an MCP server

**Time: 10 minutes.**

You connected to MCP servers other people wrote. Now write the server.

Notice what the lab's calculator server could not prove: a model can work out `2 + 2` on its own, so
a correct answer never told you the MCP call happened. Your server holds warehouse stock numbers
that the model **cannot** know. If the answer is right, the call happened.

**Write two things**, both inside the server file below:

1. The `description` for your tool. This is the MCP schema the model reads to decide when to call it.
2. The tool body: return the stock level for the requested part.

You are using the **stdio** transport, so every run starts a fresh server process. After you edit the
server, run the `%%writefile` cell **and then the client cell below it** — the client is what spawns
the new process.

In [ ]:
%%writefile warehouse_server.py
from pathlib import Path

from mcp.server import FastMCP

# Provided: the warehouse data. The model has no way to know these numbers.
STOCK = {"widget-a": 137, "widget-b": 42, "widget-c": 906}

# Provided: the server appends here every time the tool runs. This is your proof.
CALL_LOG = Path(__file__).parent / "mcp_calls.log"

mcp = FastMCP("Warehouse Server")


# TODO: replace "TODO" with a description that tells the model when to use this tool.
@mcp.tool(description="TODO")
def stock_level(part: str) -> str:
    CALL_LOG.open("a").write(f"stock_level {part}\n")   # provided: keep this line
    # TODO: return the number of units of `part` in STOCK.
    #       Handle a part that is not in STOCK too.
    raise NotImplementedError("Write the tool body, then re-run this cell and the next one.")


if __name__ == "__main__":
    mcp.run(transport="stdio")

In [ ]:
# ---- provided: connect to YOUR server over stdio and ask it a question ----
import sys
from pathlib import Path

from mcp import StdioServerParameters, stdio_client
from strands import Agent
from strands.tools.mcp import MCPClient

CALL_LOG = Path("mcp_calls.log")
CALL_LOG.unlink(missing_ok=True)          # fresh evidence for every attempt

QUESTION = "How many units of widget-a are in stock?"

warehouse_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(command=sys.executable, args=["warehouse_server.py"])
    )
)

with warehouse_client:
    mcp_tools = warehouse_client.list_tools_sync()
    print("tools your server exposes:", [t.tool_name for t in mcp_tools])

    stock_agent = Agent(
        model="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
        system_prompt="You are a warehouse assistant. Answer using the tools provided.",
        tools=mcp_tools,
    )
    mcp_answer = stock_agent(QUESTION)

In [ ]:
server_log = CALL_LOG.read_text().strip() if CALL_LOG.exists() else ""

checks = {
    "your MCP server actually received the call":
        "stock_level" in server_log,
    "the answer reports the real stock level (137)":
        "137" in str(mcp_answer),
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

print(f"\nserver-side log: {server_log or '(empty - the tool never ran)'}")
print(f"[info, not a criterion] the agent said: {str(mcp_answer)[:140]}")

print()
if all(checks.values()):
    print("Exercise complete.")
elif "stock_level" not in server_log:
    print("The tool never ran. Check the wiring: did the %%writefile cell run before the client?")
else:
    print("The tool ran but the number is wrong. Look at the tool body.")

## Stuck?

One possible solution is stored below, encoded so you do not read it by accident. Try the exercise
first: the point is the attempt, not the answer.

Run `show_solution()` to print it, or `load_solution()` to run it, then re-run the self-check.

In [ ]:
import base64

_SOLUTION = (
    "ZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpzZXJ2ZXJfY29kZSA9ICcnJ2Zyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoK"
    "ZnJvbSBtY3Auc2VydmVyIGltcG9ydCBGYXN0TUNQCgpTVE9DSyA9IHsid2lkZ2V0LWEiOiAxMzcsICJ3aWRnZXQtYiI6"
    "IDQyLCAid2lkZ2V0LWMiOiA5MDZ9CkNBTExfTE9HID0gUGF0aChfX2ZpbGVfXykucGFyZW50IC8gIm1jcF9jYWxscy5s"
    "b2ciCgptY3AgPSBGYXN0TUNQKCJXYXJlaG91c2UgU2VydmVyIikKCgpAbWNwLnRvb2woCiAgICBkZXNjcmlwdGlvbj0o"
    "CiAgICAgICAgIkxvb2sgdXAgaG93IG1hbnkgdW5pdHMgb2YgYSB3YXJlaG91c2UgcGFydCBhcmUgY3VycmVudGx5IGlu"
    "IHN0b2NrLiAiCiAgICAgICAgIlVzZSB0aGlzIHdoZW5ldmVyIHNvbWVvbmUgYXNrcyBhYm91dCBzdG9jaywgaW52ZW50"
    "b3J5IG9yIGF2YWlsYWJpbGl0eSAiCiAgICAgICAgIm9mIGEgcGFydCBzdWNoIGFzIHdpZGdldC1hLiIKICAgICkKKQpk"
    "ZWYgc3RvY2tfbGV2ZWwocGFydDogc3RyKSAtPiBzdHI6CiAgICBDQUxMX0xPRy5vcGVuKCJhIikud3JpdGUoZiJzdG9j"
    "a19sZXZlbCB7cGFydH1cXG4iKQogICAgaWYgcGFydCBub3QgaW4gU1RPQ0s6CiAgICAgICAgcmV0dXJuIGYie3BhcnR9"
    "IGlzIG5vdCBhIGtub3duIHBhcnQuIEtub3duIHBhcnRzOiB7JywgJy5qb2luKFNUT0NLKX0uIgogICAgcmV0dXJuIGYi"
    "e3BhcnR9OiB7U1RPQ0tbcGFydF19IHVuaXRzIGluIHN0b2NrIgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAg"
    "ICBtY3AucnVuKHRyYW5zcG9ydD0ic3RkaW8iKQonJycKClBhdGgoIndhcmVob3VzZV9zZXJ2ZXIucHkiKS53cml0ZV90"
    "ZXh0KHNlcnZlcl9jb2RlKQpDQUxMX0xPRy51bmxpbmsobWlzc2luZ19vaz1UcnVlKQoKd2l0aCB3YXJlaG91c2VfY2xp"
    "ZW50OgogICAgbWNwX3Rvb2xzID0gd2FyZWhvdXNlX2NsaWVudC5saXN0X3Rvb2xzX3N5bmMoKQogICAgc3RvY2tfYWdl"
    "bnQgPSBBZ2VudCgKICAgICAgICBtb2RlbD0idXMuYW50aHJvcGljLmNsYXVkZS1zb25uZXQtNC01LTIwMjUwOTI5LXYx"
    "OjAiLAogICAgICAgIHN5c3RlbV9wcm9tcHQ9IllvdSBhcmUgYSB3YXJlaG91c2UgYXNzaXN0YW50LiBBbnN3ZXIgdXNp"
    "bmcgdGhlIHRvb2xzIHByb3ZpZGVkLiIsCiAgICAgICAgdG9vbHM9bWNwX3Rvb2xzLAogICAgKQogICAgbWNwX2Fuc3dl"
    "ciA9IHN0b2NrX2FnZW50KFFVRVNUSU9OKQoKIyBUd28gdGhpbmdzIHdvcnRoIG5vdGljaW5nLiBUaGUgZGVzY3JpcHRp"
    "b24gaXMgdGhlIHdob2xlIHNjaGVtYSB0aGUgbW9kZWwgc2VlcyBmb3IgYW4KIyBNQ1AgdG9vbCwgZXhhY3RseSBsaWtl"
    "IGEgZG9jc3RyaW5nIGZvciBhIEB0b29sIGZ1bmN0aW9uLiBBbmQgYmVjYXVzZSBzdGRpbyBzdGFydHMgYQojIG5ldyBz"
    "ZXJ2ZXIgcHJvY2VzcyBvbiBldmVyeSBjb25uZWN0aW9uLCBlZGl0aW5nIHRoZSBmaWxlIGFuZCByZS1ydW5uaW5nIHRo"
    "ZSBjbGllbnQgaXMKIyBlbm91Z2ggLSB0aGVyZSBpcyBubyBzdGFsZSBzZXJ2ZXIgdG8gcmVzdGFydC4="
)


def show_solution() -> None:
    """Print one possible solution to the exercise."""
    print(base64.b64decode(_SOLUTION).decode())


def load_solution() -> None:
    """Run the reference solution, then re-run the self-check cell to see it pass."""
    exec(base64.b64decode(_SOLUTION).decode(), globals())


print("Ready. Call show_solution() to read it, or load_solution() to run it.")